Copyright© Gergely Molnar, CNRS, INSA of Lyon, LaMCoS, UMR5259

In [ ]:
# Copyright© Gergely Molnar, CNRS, INSA of Lyon, LaMCoS, UMR5259
#
# gergely.molnar@insa-lyon.fr
# http://www.molnar-research.com
#
# This software is a computer program whose purpose is to calculate
# the fracture resistance of elastic-brittle solids in Abaqus under
# static and dynamic conditions.
#
# This software is governed by the CeCILL-B license under French law and
# abiding by the rules of distribution of free software. You can  use, 
# modify and/or redistribute the software under the terms of the CeCILL-B
# license as circulated by CEA, CNRS and INRIA at the following URL
# "http://www.cecill.info". 
#
# As a counterpart to the access to the source code and  rights to copy,
# modify and redistribute granted by the license, users are provided only
# with a limited warranty  and the software's author,  the holder of the
# economic rights,  and the successive licensors  have only  limited
# liability. 
#
# In this respect, the user's attention is drawn to the risks associated
# with loading,  using,  modifying and/or developing or reproducing the
# software by the user in light of its specific status of free software,
# that may mean  that it is complicated to manipulate,  and  that  also
# therefore means  that it is reserved for developers  and  experienced
# professionals having in-depth computer knowledge. Users are therefore
# encouraged to load and test the software's suitability as regards their
# requirements in conditions enabling the security of their systems and/or 
# data to be ensured and,  more generally, to use and operate it in the 
# same conditions as regards security. 
#
# The fact that you are presently reading this means that you have had
# knowledge of the CeCILL-B
#

# Simple 2D Finite Element Module

## Create initial mesh and Material properties


In [ ]:
# Load python libreries
import numpy as np
import matplotlib.pyplot as plt
from scipy.sparse import coo_matrix

In [ ]:
# ---------- Construct model ----------

# Nodes
coor = np.array([
    [0,  0],
    [0,  5],
    [5,  0],
    [5,  5],
    [10, 0],
    [10, 5]])

# Connectivity
connec = np.array([
    [1, 0, 2, 3],
    [3, 2, 4, 5]])

# Material parameters
E = 70.0
nu = 0.3

# Boundary conditions
# Dirichlet boundary conditions
# Constrained DOFs
sup = [0, 1, 2]

# Prescribed displacements
dispbond = [0, 0, 0]


# Neumann boundary conditions
# DOFs
supF = [8, 10]

# Force values
Fval = [100, 100]

## Plot nodes and elements

In [ ]:
def plot_mesh(coor, connec, sup, dispbond, supF, Fval):

    # Plot elements
    for elem in connec:
        nodes = elem[[0, 1, 2, 3, 0]]

        plt.plot(coor[nodes, 0], coor[nodes, 1], 'k-')

    # Plot nodes
    plt.plot(coor[:, 0], coor[:, 1], 'ko')

    # Node numbers
    for i, (x, y) in enumerate(coor):
        plt.text(x-0.1, y+0.1, str(i), fontsize=12,
                 horizontalalignment='right',
                 verticalalignment='bottom')

    # Dirichlet boundary conditions
    for dof in sup:
        node = dof // 2
        direction = dof % 2

        x, y = coor[node]

        if direction == 0:  # x direction
            plt.plot(x-0.25, y, marker='>', markersize=10, color='blue')
        else:               # y direction
            plt.plot(x, y-0.25, marker='^', markersize=10, color='blue')

    # Neumann boundary conditions
    for dof, force in zip(supF, Fval):
        node = dof // 2
        direction = dof % 2

        x, y = coor[node]

        if direction == 0:  # Force in x direction
            plt.arrow(x, y, force/100, 0,
                      head_width=0.15, head_length=0.25,
                      length_includes_head=True, color='red')
        else:               # Force in y direction
            plt.arrow(x, y, 0, force/100,
                      head_width=0.15, head_length=0.25,
                      length_includes_head=True, color='red')

    plt.axis('equal')
    plt.xlabel('x')
    plt.ylabel('y')
    plt.title('Finite Element Geometry')
    plt.show()

In [ ]:
plot_mesh(coor, connec, sup, dispbond, supF, Fval)

## Shape functions

For this example, we will use fully integrated linear quadrilateral elements.

<img src="images/shapeFun.png" width="500">

The corresponding shape functions are

$$
\begin{aligned}
N_1 &= \frac{1}{4}(1-\xi)(1-\eta),\\
N_2 &= \frac{1}{4}(1+\xi)(1-\eta),\\
N_3 &= \frac{1}{4}(1+\xi)(1+\eta),\\
N_4 &= \frac{1}{4}(1-\xi)(1+\eta).
\end{aligned}
$$

**Try to code one of these shape functions!**

Then evaluate it at each of the four nodes in the local coordinate system. What values do you obtain?

In [ ]:
# ------------------------------------------------
# Write down the expression for one of the shape functions Ni

xi = [-1, -1]    # Change these values

N = 

print("N =", N)

# ------------------------------------------------


This can be summarized in a function as follows:

In [ ]:
def shapefun(xi):
    N = [
        1/4 * (1-xi[0]) * (1-xi[1]),
        1/4 * (1+xi[0]) * (1-xi[1]),
        1/4 * (1+xi[0]) * (1+xi[1]),
        1/4 * (1-xi[0]) * (1+xi[1])
    ]

    return N

Now evaluate the shape functions at a random location $(\xi,\eta)$ within the domain $[-1,1]\times[-1,1]$.

**What do you observe?**

In [ ]:
# ------------------------------------------------
xi=[-0.5, -0.5]

N = 

print(N)
# ------------------------------------------------

## Local spatial derivatives

Strain is defined in terms of the first spatial derivatives of the displacement field, which is our primary degree of freedom. Therefore, we need to calculate the derivatives of each shape function in both directions.

**Try calculating the derivatives of $N_1$ with respect to $\xi$ and $\eta$. Then code them and evaluate them at different locations. What do you observe?**

In [ ]:
# ------------------------------------------------
# Calculate the derivatives of N1

xi = [-1, -1]    # Change these values

dN1dxi  = 
dN1deta = 

print("dN1/dxi  =", dN1dxi)
print("dN1/deta =", dN1deta)

# ------------------------------------------------




This can be summarized in a function as follows:

In [ ]:
def shapefun_derivative(xi):
    dNdxi = np.array([
        [-1/4 * (1-xi[1]), -1/4 * (1-xi[0])],
        [ 1/4 * (1-xi[1]), -1/4 * (1+xi[0])],
        [ 1/4 * (1+xi[1]),  1/4 * (1+xi[0])],
        [-1/4 * (1+xi[1]),  1/4 * (1-xi[0])]
    ])
    
    return dNdxi

## Numerical integration (Gauss–Legendre quadrature)

The four local integration points for the $2\times2$ Gauss–Legendre quadrature are located at

$$
(\xi,\eta) =
\left(\pm\frac{1}{\sqrt{3}},\pm\frac{1}{\sqrt{3}}\right),
$$

with the corresponding integration weights

$$
w_i = 1, \qquad i=1,\ldots,4.
$$

In [ ]:
# Local integration point coordinates
sqr3 = 1 / np.sqrt(3)

xii = np.array([
    [-sqr3, -sqr3],
    [ sqr3, -sqr3],
    [ sqr3,  sqr3],
    [-sqr3,  sqr3]
])

print(xii)

wi = np.array([1, 1, 1, 1])

print(wi)

## Material model

In this simple example, we will use a linear elastic Hooke's law under plane strain conditions.

In [ ]:
# Material stiffness matrix - plane strain
C = E / ((1 + nu) * (1 - 2*nu)) * np.array([
    [1-nu, nu,   0],
    [nu,   1-nu, 0],
    [0,    0,    (1-2*nu)/2]])

print(C)

## Finite Element Algorithm

## Finite element algorithm

After completing the initialization steps, we can now start building the finite element algorithm.

The main steps are:

1. **Initialize the global matrices and vectors**

   Create the global stiffness matrix $\mathbf{K}$, residual vector $\mathbf{R}$, and displacement vector $\mathbf{U}$.

2. **Loop over all finite elements**

   For each element, calculate the elementary residual vector $\mathbf{R}^e$ and stiffness matrix $\mathbf{K}^e$.

3. **Loop over the integration points of each element**

   At each integration point:

   a. Calculate the transformation from local $(\xi,\eta)$ to global $(x,y)$ coordinates using the Jacobian matrix
   
   $$
   \mathbf{J} = \frac{\partial(x,y)}{\partial(\xi,\eta)}.
   $$

   b. Transform the shape-function derivatives to the global coordinate system and construct the strain-displacement matrix $\mathbf{B}$.

   c. Calculate the strain
   
   $$
   \boldsymbol{\varepsilon} = \mathbf{B}\mathbf{U}^e.
   $$

   d. Calculate the stress using the constitutive law
   
   $$
   \boldsymbol{\sigma} = \mathbf{C}\boldsymbol{\varepsilon}.
   $$

   e. Calculate the contribution to the elementary residual vector
   
   $$
   \mathbf{R}^e =
   \int_{\Omega^e}
   \mathbf{B}^T\boldsymbol{\sigma}\,d\Omega.
   $$

   f. Calculate the contribution to the elementary stiffness matrix
   
   $$
   \mathbf{K}^e =
   \int_{\Omega^e}
   \mathbf{B}^T\mathbf{C}\mathbf{B}\,d\Omega.
   $$

4. **Assemble the elementary contributions**

   Assemble $\mathbf{R}^e$ and $\mathbf{K}^e$ into the global residual vector $\mathbf{R}$ and stiffness matrix $\mathbf{K}$.

5. **Apply the boundary conditions**

   Apply the prescribed Dirichlet and Neumann boundary conditions.

6. **Solve the linear system**

   Solve for the unknown nodal displacements:

   $$
   \mathbf{K}\mathbf{U} = \mathbf{F}.
   $$

7. **Check convergence**

   Verify whether the convergence criterion is satisfied. If not, update the solution and repeat the procedure.



## Initialization

In [ ]:
nelem = connec.shape[0]
nnode = coor.shape[0]

# Initializing main vectors and matrices
F = np.zeros(2*nnode)
U = np.zeros(2*nnode)
Fext = np.zeros(2*nnode)

# Positions of the stiffness components (sparce matrix)
Ivect = []
Jvect = []
Kvect = []

## Elementary calculations

The following quantities are unique to each element and do not change between integration points.

In [ ]:
# Do it for example for the first elment:
e = 0

In [ ]:
# ------------------------------------------------
# Initilize local residue
Floc = 





# Initilize local stiffness
Kloc =

# ------------------------------------------------

**Determine which global degrees of freedom correspond to each local node**

In [ ]:
# Global indeces
DOFLoc = np.array([dof for n in connec[e, :] for dof in (2*n, 2*n+1)])
print(DOFLoc)

**Extract the local nodal displacements from the global displacement vector**

In [ ]:
# ------------------------------------------------
# Local nodal displacements
Uloc = 






print('Uloc = ',Uloc)
# ------------------------------------------------

**Extract local coordinates** ($\mathbf{X}^e$)

In [ ]:
# ------------------------------------------------
# Local coordinates
COORDS = 






print('COORDS = ',COORDS)
# ------------------------------------------------

## Start summarizing on each integration point

**Start on the first integration point**

In [ ]:
ipnt = 0

**Calcaulate local coordinates of the first integration point:**

In [ ]:
# ------------------------------------------------
xi = 



print('xi = ',xi)
# ------------------------------------------------

**Calculate values of the shape functions on this integration point:**

In [ ]:
# ------------------------------------------------
N = 





print('N = ',N)
# ------------------------------------------------

**Calculate local spatial derivatives:**

In [ ]:
# ------------------------------------------------
dNdxi =





print('dNdxi = ',dNdxi)
# ------------------------------------------------

### Calculate the Jacobian matrix

The Jacobian matrix defines the transformation between the local coordinates $(\xi,\eta)$ and the global coordinates $(x,y)$:

$$
\mathbf{J}
=
\begin{bmatrix}
\dfrac{\partial x}{\partial \xi} & \dfrac{\partial y}{\partial \xi} \\[6pt]
\dfrac{\partial x}{\partial \eta} & \dfrac{\partial y}{\partial \eta}
\end{bmatrix}.
$$

Its components are calculated from the nodal coordinates and the derivatives of the shape functions:

$$
\begin{aligned}
\frac{\partial x}{\partial \xi}
&= \sum_{i=1}^{n} x_i \frac{\partial N_i}{\partial \xi},
&
\frac{\partial y}{\partial \xi}
&= \sum_{i=1}^{n} y_i \frac{\partial N_i}{\partial \xi}, \\[6pt]
\frac{\partial x}{\partial \eta}
&= \sum_{i=1}^{n} x_i \frac{\partial N_i}{\partial \eta},
&
\frac{\partial y}{\partial \eta}
&= \sum_{i=1}^{n} y_i \frac{\partial N_i}{\partial \eta}.
\end{aligned}
$$

In matrix form, this can be written compactly as

$$
\boxed{\mathbf{J} =
\left(\frac{\partial \mathbf{N}}{\partial \boldsymbol{\xi}}\right)^T
\mathbf{X}^e}
$$

In [ ]:
# ------------------------------------------------
# Jacobien
J = 




print('J = ',J)
# ------------------------------------------------

**Calculate its determinant:** ${J}=\det\left({\mathbf{J}}\right)$

In [ ]:
# ------------------------------------------------
DTM = 





print('DTM = ',DTM)
# ------------------------------------------------

**Transform the shape-function derivatives to the global coordinate system**

$$
\boxed{
\frac{\partial \mathbf{N}}{\partial \mathbf{X}}
=
\frac{\partial \mathbf{N}}{\partial \boldsymbol{\xi}}
\mathbf{J}^{-1}
}
$$

In [ ]:
# ------------------------------------------------
# Transform the shape-function derivatives to the global coordinate system
dNdX = 



# ------------------------------------------------

### Calculate the strain–displacement matrix $\mathbf{B}$

The displacement field inside the element is interpolated from the nodal displacements using the shape functions:

$$
u(x,y)=\sum_{i=1}^{n}N_i(x,y)u_i,
\qquad
v(x,y)=\sum_{i=1}^{n}N_i(x,y)v_i.
$$

For small strains,

$$
\varepsilon_x=\frac{\partial u}{\partial x},
\qquad
\varepsilon_y=\frac{\partial v}{\partial y},
\qquad
\gamma_{xy}=
\frac{\partial u}{\partial y}
+
\frac{\partial v}{\partial x}.
$$

Using the finite-element interpolation,

$$
\begin{aligned}
\varepsilon_x
&=
\sum_{i=1}^{n}
\frac{\partial N_i}{\partial x}u_i,\\[4pt]
\varepsilon_y
&=
\sum_{i=1}^{n}
\frac{\partial N_i}{\partial y}v_i,\\[4pt]
\gamma_{xy}
&=
\sum_{i=1}^{n}
\left(
\frac{\partial N_i}{\partial y}u_i
+
\frac{\partial N_i}{\partial x}v_i
\right).
\end{aligned}
$$

These equations can be written in matrix form as

$$
\boxed{
\boldsymbol{\varepsilon}
=
\mathbf{B}\mathbf{U}^e
}
$$

where

$$
\boldsymbol{\varepsilon}
=
\begin{bmatrix}
\varepsilon_x\\
\varepsilon_y\\
\gamma_{xy}
\end{bmatrix},
\qquad
\mathbf{U}^e
=
\begin{bmatrix}
u_1 & v_1 & u_2 & v_2 & \cdots & u_n & v_n
\end{bmatrix}^{T},
$$

and the strain–displacement matrix is

$$
\boxed{
\mathbf{B}
=
\begin{bmatrix}
\dfrac{\partial N_1}{\partial x} & 0 &
\dfrac{\partial N_2}{\partial x} & 0 &
\cdots &
\dfrac{\partial N_n}{\partial x} & 0
\\[6pt]
0 & \dfrac{\partial N_1}{\partial y} &
0 & \dfrac{\partial N_2}{\partial y} &
\cdots &
0 & \dfrac{\partial N_n}{\partial y}
\\[6pt]
\dfrac{\partial N_1}{\partial y} &
\dfrac{\partial N_1}{\partial x} &
\dfrac{\partial N_2}{\partial y} &
\dfrac{\partial N_2}{\partial x} &
\cdots &
\dfrac{\partial N_n}{\partial y} &
\dfrac{\partial N_n}{\partial x}
\end{bmatrix}.
}
$$

In [ ]:
# ------------------------------------------------
# Strain-displacement matrix
B = 




# ------------------------------------------------


**Calcualte local strain on the integration point**

In [ ]:
# ------------------------------------------------
# Strain
eps =




print('esp = ',eps)
# ------------------------------------------------

**Calcualte local stresses on the integration point**

In [ ]:
# ------------------------------------------------
# Stress
sig =





print('sig = ',sig)
# ------------------------------------------------

### Calculate the local internal force vector

The local internal force vector is obtained by integrating the stresses over the element:

$$
\boxed{
\mathbf{F}_{\mathrm{int}}^e
=
\int_{\Omega^e}
\mathbf{B}^T \boldsymbol{\sigma}\,d\Omega^e
}
$$

Using Gauss–Legendre quadrature, the integral is approximated as

$$
\boxed{
\mathbf{F}_{\mathrm{int}}^e
=
\sum_{n=1}^{p}
\mathbf{B}_n^T
\boldsymbol{\sigma}_n
\det(\mathbf{J}_n)
w_n
}
$$

where $p$ is the number of integration points and $w_n$ is the corresponding integration weight.

In [ ]:
# ------------------------------------------------
# Local internal force vector
Floc =






# ------------------------------------------------

### Calculate the local stiffness matrix

The local stiffness matrix is obtained by integrating the material stiffness over the element:

$$
\boxed{
\mathbf{K}^e
=
\int_{\Omega^e}
\mathbf{B}^T
\mathbf{C}
\mathbf{B}
\,d\Omega^e
}
$$

Using Gauss–Legendre quadrature, the integral is approximated as

$$
\boxed{
\mathbf{K}^e
=
\sum_{n=1}^{p}
\mathbf{B}_n^T
\mathbf{C}
\mathbf{B}_n
\det(\mathbf{J}_n)
w_n
}
$$

where $p$ is the number of integration points and $w_n$ is the corresponding integration weight.

In [ ]:
# ------------------------------------------------
# Local stiffness matrix
Kloc =




# ------------------------------------------------

## Summary in a python function

In [ ]:
def element_matrix(e, coor, connec, U, xii, wi, C):

    # Initialize local internal force and stiffness
    Floc = np.zeros(8)
    Kloc = np.zeros((8, 8))

    # Initialize strain and stress at integration points
    epsOut = np.zeros((4, 3))
    sigOut = np.zeros((4, 3))
    
    # Global DOFs corresponding to the local nodes
    DOFLoc = np.array([dof for n in connec[e, :] for dof in (2*n, 2*n+1)])

    # Local nodal coordinates
    COORDS = coor[connec[e, :], :]

    # Local nodal displacements
    Uloc = U[DOFLoc]

    # Loop over integration points
    for ipnt in range(4):

        # Local integration point coordinates
        xi = xii[ipnt, :]

        # Shape functions
        N = shapefun(xi)

        # Shape-function derivatives in local coordinates
        dNdxi = shapefun_derivative(xi)

        # Jacobian
        J = dNdxi.T @ COORDS

        # Determinant of the Jacobian
        DTM = np.linalg.det(J)

        # Transform derivatives to global coordinates
        dNdX = dNdxi @ np.linalg.inv(J)

        # Strain-displacement matrix
        B = np.array([
            [dNdX[0,0], 0,          dNdX[1,0], 0,          dNdX[2,0], 0,          dNdX[3,0], 0],
            [0,          dNdX[0,1], 0,          dNdX[1,1], 0,          dNdX[2,1], 0,          dNdX[3,1]],
            [dNdX[0,1], dNdX[0,0], dNdX[1,1], dNdX[1,0], dNdX[2,1], dNdX[2,0], dNdX[3,1], dNdX[3,0]]
        ])

        # Strain
        eps = B @ Uloc
        epsOut[ipnt, :] = eps

        # Stress
        sig = C @ eps
        sigOut[ipnt, :] = sig        

        # Local internal force
        Floc += B.T @ sig * DTM * wi[ipnt]

        # Local tangent stiffness
        Kloc += B.T @ C @ B * DTM * wi[ipnt]

    return Floc, Kloc, DOFLoc, epsOut, sigOut


## Assembly of the global system

Once the local stiffness matrix $\mathbf{K}^e$ and internal force vector
$\mathbf{F}_{\mathrm{int}}^e$ have been calculated, they must be assembled into
the global system.

Each local degree of freedom corresponds to a global degree of freedom through
the vector `DOFLoc`.

The assembly consists of adding each local contribution to its corresponding
position in the global system:

$$
\boxed{
F_{\mathrm{int},I}
\mathrel{+}=
F_{\mathrm{int},i}^{e}
}
$$

and

$$
\boxed{
K_{IJ}
\mathrel{+}=
K_{ij}^{e}
}
$$

where

$$
I = \mathrm{DOFLoc}[i],
\qquad
J = \mathrm{DOFLoc}[j].
$$

The contributions are **added** because several elements can share the same
global degrees of freedom.

In [ ]:
def assemble_system(coor, connec, U, xii, wi, C):

    # Number of nodes and elements
    nnode = coor.shape[0]
    nelem = connec.shape[0]

    # Initialize global internal force vector
    F = np.zeros(2*nnode)

    # Initialize sparse stiffness storage
    Ivect = []
    Jvect = []
    Kvect = []

    # Loop over elements
    for e in range(nelem):

        # Calculate elementary quantities
        Floc, Kloc, DOFLoc, eps, sig = element_matrix(
            e, coor, connec, U, xii, wi, C
        )

        # Assemble
        for i in range(8):

            I = DOFLoc[i]

            # Internal force vector
            F[I] += Floc[i]

            # Stiffness matrix
            for j in range(8):

                J = DOFLoc[j]

                Ivect.append(I)
                Jvect.append(J)
                Kvect.append(Kloc[i, j])

    # Construct global sparse stiffness matrix
    K = coo_matrix(
        (Kvect, (Ivect, Jvect)),
        shape=(2*nnode, 2*nnode)
    ).tocsr()

    return F, K

## Apply boundary conditions and external forces

Before solving the system, the prescribed displacements (Dirichlet boundary conditions) and external forces (Neumann boundary conditions) must be applied.

For simplicity, we first convert the sparse global stiffness matrix into a full matrix.

For each prescribed displacement $U_J=\bar{U}_J$, its contribution is transferred to the right-hand side:

$$
\mathbf{F} \leftarrow \mathbf{F} - \mathbf{K}_{:,J}\bar{U}_J.
$$

The corresponding row and column of the stiffness matrix are then set to zero, the diagonal term is replaced by one, and the prescribed displacement is imposed on the right-hand side.

Finally, the external nodal forces are introduced and the global residual is calculated as

$$
\boxed{
\mathbf{R}
=
\mathbf{F}_{\mathrm{ext}}
-
\mathbf{F}_{\mathrm{int}}
}
$$

In [ ]:
def apply_boundary_conditions(K, F, sup, dispbond, supF, Fval):

    # Convert sparse stiffness matrix to full matrix
    K = K.toarray()

    # Initialize external force vector
    Fext = np.zeros(len(F))

    # Apply prescribed displacements
    for i in range(len(sup)):
        J = sup[i]
        F -= K[:, J] * dispbond[i]

    F[sup] = dispbond

    # Modify stiffness matrix
    K[sup, :] = 0
    K[:, sup] = 0

    for i in range(len(sup)):
        K[sup[i], sup[i]] = 1

    # Apply external forces
    Fext[supF] = Fval

    # Calculate residual
    F = Fext - F

    return F, K

In [ ]:
# Execute assembly and apply boundary conditions
F, K = assemble_system(coor, connec, U, xii, wi, C)
F, K = apply_boundary_conditions(K, F, sup, dispbond, supF, Fval)

## Solve the system

Once the global stiffness matrix and force vector have been assembled and the boundary conditions applied, the nodal displacements are obtained by solving

$$
\boxed{
\mathbf{K}\,\Delta\mathbf{U}=\mathbf{F}
}
$$

The displacement increment is then added to the current displacement vector:

$$
\boxed{
\mathbf{U} \leftarrow \mathbf{U}+\Delta\mathbf{U}
}
$$

In [ ]:
# Solve linear equation system
dU = np.linalg.solve(K, F)

In [ ]:
# Plot the deformed shape
# Reorder dU into coor format
dUcoor = dU.reshape((-1, 2))

#plot structure with new positions
plot_mesh(coor+dUcoor, connec, sup, dispbond, supF, Fval)

## Verify convergence

For an iterative solution, convergence must be verified before accepting the results. This can be done by checking that the residual forces or the displacement increment are sufficiently small:

$$
\|\mathbf{R}\| < \mathrm{tol}
\qquad \text{or} \qquad
\|\Delta\mathbf{U}\| < \mathrm{tol}.
$$

Once convergence is reached, the nodal displacements are considered to be the solution, and secondary variables such as **strains** and **stresses** can be evaluated.

In [ ]:
# Update displacements and verify convergence
F, K = assemble_system(coor, connec, U+dU, xii, wi, C)
F, K = apply_boundary_conditions(K, F, sup, dispbond, supF, Fval)
dU2 = np.linalg.solve(K, F)


print('Maximum displacement change : ',max(dU2))
print('Maximum residue: ',max(F))

## Calculate secondary variables

The nodal displacements are the primary unknowns of the finite element problem. Once they are known, secondary variables such as **strains** and **stresses** can be calculated within each element:

$$
\boldsymbol{\varepsilon}=\mathbf{B}\mathbf{U}^e,
\qquad
\boldsymbol{\sigma}=\mathbf{C}\boldsymbol{\varepsilon}.
$$

These quantities are typically evaluated at the integration points and can then be post-processed for visualization and analysis.

In [ ]:
# Calcualte stresses and strains with the new solution in the first element
e = 0
Floc, Kloc, DOFLoc, eps, sig = element_matrix(e, coor, connec, U+dU, xii, wi, C)

print('sig = ',sig)
